In [1]:
# Tanı: sys.path ve dosya yapısını kontrol et
import sys, os, glob
print('sys.path:')
for p in sys.path:
    print(p)
print('\nİçerik (src/envs):', glob.glob('../src/envs/*'))
print('\nİçerik (src):', glob.glob('../src/*'))
print('\nÇalışma dizini:', os.getcwd())

sys.path:
/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python39.zip
/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9
/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/lib-dynload

/Users/berkaygurkan/pytorch_m1_env/lib/python3.9/site-packages

İçerik (src/envs): ['../src/envs/__init__.py', '../src/envs/wrappers.py']

İçerik (src): ['../src/__init__.py', '../src/agents', '../src/envs']

Çalışma dizini: /Users/berkaygurkan/Desktop/Thesis/RL/notebooks


In [ ]:
import gymnasium as gym
import time
import numpy as np

# --- MDP ve Gymnasium Bağlantısı ---
# Gymnasium, RL problemlerini standart bir arayüzle sunar.
# Bu arayüz, MDP tanımının pratik bir uygulamasıdır.

# Ortamı oluşturalım. "CartPole-v1" en klasik RL problemlerinden biridir.
# render_mode="human" ile ortamı görsel olarak görebiliriz.
env = gym.make("CartPole-v1", render_mode="human")

# S: Durum Uzayı (Observation Space)
print("Durum (Gözlem) Uzayı:", env.observation_space)
print("Örnek bir durum:", env.observation_space.sample())
print("-" * 30)

# A: Eylem Uzayı (Action Space)
print("Eylem Uzayı:", env.action_space)
print("Örnek bir eylem:", env.action_space.sample())
print("-" * 30)

# --- Ajan-Çevre Döngüsü ---
# Bir bölümü (episode) simüle edelim.
# Ajanımız şimdilik tamamen rastgele eylemler seçecek.

# Ortamı sıfırla ve ilk durumu (observation) al. Bu, S_0'dır.
observation, info = env.reset(seed=42)
total_reward = 0

for _ in range(5):
    # Ortamı görselleştir
    env.render()
    
    # A: Rastgele bir eylem seç
    action = env.action_space.sample()
    
    # Ajan eylemi gerçekleştirir ve çevreden geri bildirim alır
    # Bu, MDP'nin P ve R fonksiyonlarının bir sonucudur.
    # step() fonksiyonu bize 5 değer döndürür:
    # next_observation (s'): Bir sonraki durum
    # reward (r): Alınan ödül
    # terminated: Bölüm bitti mi? (örn. hedefe ulaşıldı veya ajan "öldü")
    # truncated: Bölüm zaman limitinden dolayı mı bitti?
    # info: Ekstra bilgi (genellikle hata ayıklama için)
    observation, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    
    # Eğer bölüm bittiyse (terminated veya truncated), döngüyü kır ve baştan başla
    if terminated or truncated:
        print(f"Bölüm bitti. Toplam Ödül: {total_reward}")
        observation, info = env.reset()
        total_reward = 0
        time.sleep(1) # Yeni bölümden önce kısa bir bekleme

# Simülasyon bittiğinde ortamı kapat
env.close()

Durum (Gözlem) Uzayı: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Örnek bir durum: [ 4.3236575 -0.7427793  0.4186096 -1.3974428]
------------------------------
Eylem Uzayı: Discrete(2)
Örnek bir eylem: 1
------------------------------


: 

In [2]:
import gymnasium as gym
import sys
import os

# Proje kök dizinini sys.path'e ekle (notebook'u /notebooks klasöründen çalıştırıyorsan gereklidir)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# Artık src.envs.wrappers importu çalışacaktır
from src.envs.wrappers import RewardShapeWrapper

# --- Sarmalayıcıyı Kullanalım ---
# Önce normal bir ortam oluşturalım
base_env = gym.make("CartPole-v1") 

# Şimdi de bu ortamı kendi sarmalayıcımızla "saralım"
# Bu sefer parametreleri de belirtebiliriz.
wrapped_env = RewardShapeWrapper(base_env, angle_threshold=0.05, penalty=1.0)

# Ortamları sıfırlayalım
obs_base, _ = base_env.reset(seed=123)
obs_wrapped, _ = wrapped_env.reset(seed=123)

action = 1 

# Sopanın açısını ceza sınırını aşana kadar aynı eylemi uygulayalım
for _ in range(10):
    obs_base, reward_base, _, _, _ = base_env.step(action)
    obs_wrapped, reward_wrapped, _, _, _ = wrapped_env.step(action)

    print(f"Açı: {obs_wrapped[2]:.4f} | Orijinal Ödül: {reward_base} | Sarmalanmış Ödül: {reward_wrapped}")
    
    if abs(obs_wrapped[2]) > 0.05: # Eşiği yeni değerimize göre güncelledik
        print(">>> Ceza eşiği aşıldı! Ödül farkına dikkat et. <<<")
        break

base_env.close()
wrapped_env.close()

Açı: -0.0353 | Orijinal Ödül: 1.0 | Sarmalanmış Ödül: 1.0
Açı: -0.0667 | Orijinal Ödül: 1.0 | Sarmalanmış Ödül: 0.0
>>> Ceza eşiği aşıldı! Ödül farkına dikkat et. <<<
